In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from keras_unet_collection import models
import albumentations as A
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

images = np.load("/content/drive/MyDrive/unetimages.npy")
masks  = np.load("/content/drive/MyDrive/unetmasks.npy")

In [ ]:
x_train, x_temp, y_train, y_temp = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42
)
print(len(x_train), len(x_val), len(x_test))

#Axial Attention Layer

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class AxialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super(AxialAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        channels = input_shape[-1]

        # Dense layers for height attention
        self.q_h = layers.Dense(channels)
        self.k_h = layers.Dense(channels)
        self.v_h = layers.Dense(channels)

        # Dense layers for width attention
        self.q_w = layers.Dense(channels)
        self.k_w = layers.Dense(channels)
        self.v_w = layers.Dense(channels)

        super(AxialAttention, self).build(input_shape)

    def call(self, x):
        """
        x: (B, H, W, C)
        """
        B, H, W, C = tf.unstack(tf.shape(x))

        # ---- Height-wise attention ----
        # (B, H, W, C) -> (B, H, C) by averaging over width
        h_feat = tf.reduce_mean(x, axis=2)  # (B, H, C)

        q_h = self.q_h(h_feat)  # (B, H, C)
        k_h = self.k_h(h_feat)  # (B, H, C)
        v_h = self.v_h(h_feat)  # (B, H, C)

        # Compute attention along height
        attn_scores_h = tf.matmul(q_h, k_h, transpose_b=True)  # (B, H, H)
        attn_weights_h = tf.nn.softmax(attn_scores_h, axis=-1) # (B, H, H)
        out_h = tf.matmul(attn_weights_h, v_h)                 # (B, H, C)

        # Broadcast back: (B, H, 1, C) -> (B, H, W, C)
        out_h = tf.expand_dims(out_h, axis=2)
        out_h = tf.tile(out_h, [1, 1, W, 1])

        # ---- Width-wise attention ----
        # (B, H, W, C) -> (B, W, C) by averaging over height
        w_feat = tf.reduce_mean(x, axis=1)  # (B, W, C)

        q_w = self.q_w(w_feat)
        k_w = self.k_w(w_feat)
        v_w = self.v_w(w_feat)

        attn_scores_w = tf.matmul(q_w, k_w, transpose_b=True)  # (B, W, W)
        attn_weights_w = tf.nn.softmax(attn_scores_w, axis=-1) # (B, W, W)
        out_w = tf.matmul(attn_weights_w, v_w)                 # (B, W, C)

        # Broadcast back: (B, 1, W, C) -> (B, H, W, C)
        out_w = tf.expand_dims(out_w, axis=1)
        out_w = tf.tile(out_w, [1, H, 1, 1])

        # Combine height + width attention and add residual connection
        out = out_h + out_w
        return x + out  # residual connection

    def get_config(self):
        base_config = super(AxialAttention, self).get_config()
        return base_config

#UNet + Axial Attention in Bottleneck

In [ ]:
def conv_block(x, filters, dropout_rate=0.0):
    x = layers.Conv2D(filters, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if dropout_rate > 0.0:
        x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(filters, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def build_axial_unet(input_shape=(256,256,1)):
    inputs = layers.Input(input_shape)

    # ----- Encoder -----
    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D((2,2))(c1)

    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D((2,2))(c2)

    c3 = conv_block(p2, 256)
    p3 = layers.MaxPooling2D((2,2))(c3)

    c4 = conv_block(p3, 512)
    p4 = layers.MaxPooling2D((2,2))(c4)

    # ----- Bottleneck -----
    bn = conv_block(p4, 1024, dropout_rate=0.2)

    # 🔥 Axial Attention here
    bn = AxialAttention()(bn)

    # ----- Decoder -----
    u4 = layers.UpSampling2D((2,2))(bn)
    u4 = layers.Concatenate()([u4, c4])
    c5 = conv_block(u4, 512)

    u3 = layers.UpSampling2D((2,2))(c5)
    u3 = layers.Concatenate()([u3, c3])
    c6 = conv_block(u3, 256)

    u2 = layers.UpSampling2D((2,2))(c6)
    u2 = layers.Concatenate()([u2, c2])
    c7 = conv_block(u2, 128)

    u1 = layers.UpSampling2D((2,2))(c7)
    u1 = layers.Concatenate()([u1, c1])
    c8 = conv_block(u1, 64)

    outputs = layers.Conv2D(1, (1,1), activation="sigmoid")(c8)

    model = Model(inputs, outputs, name="unet_axial_attention")
    return model


In [ ]:
axial_unet = build_axial_unet(input_shape=(256,256,1))
axial_unet.summary()

In [ ]:
def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    dice = dice_coefficient(y_true, y_pred)
    dice_loss = 1.0 - dice
    return 0.5 * bce + 0.5 * dice_loss


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

axial_unet.compile(
    optimizer=optimizer,
    loss=bce_dice_loss,              # or bce_dice_loss
    metrics=['accuracy', iou_metric, dice_coefficient]
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/best_axial_unet.h5',  # change path for Colab/GDrive
    monitor='val_loss',  # or 'val_dice_coefficient' if defined and you want to maximize
    save_best_only=True,
    verbose=1
)

callbacks = [
    checkpoint,
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6)
]


In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=10, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.2),
], additional_targets={'mask':'mask'})


In [ ]:
def tf_augment(img, mask):
    def _aug_fn(img_np, mask_np):
        # img_np float32 0-1 => convert to uint8 0-255
        img_uint8 = (img_np * 255.0).astype(np.uint8)
        mask_uint8 = (mask_np * 255.0).astype(np.uint8)

        aug = train_transform(image=img_uint8, mask=mask_uint8)

        img_out = aug['image'].astype(np.float32) / 255.0
        mask_out = (aug['mask'] > 127).astype(np.float32)

        return img_out, mask_out

    img_a, mask_a = tf.numpy_function(_aug_fn, [img, mask], [tf.float32, tf.float32])
    img_a.set_shape((256, 256, 1))
    mask_a.set_shape((256, 256, 1))

    return img_a, mask_a

def preprocess(img, mask):
    # identity if already float32 0..1, but ensure dtypes
    img = tf.cast(img, tf.float32)
    mask = tf.cast(mask > 0, tf.float32)
    return img, mask

BATCH_SIZE = 4
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(200).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_dataset = val_dataset.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
history = axial_unet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=callbacks
)


In [ ]:
results = axial_unet.evaluate(x_test, y_test, verbose=1)
print("Test Loss:", results[0])
print("Test Accuracy:", results[1])
print("Test IoU:", results[2])
print("Test Dice:", results[3])